# Module 6: Govern model and tool calls
> ~30 min · optional

In this module, policies you set in LangSmith govern the agent without changing its code. The **LLM Gateway** sits between the agent and the model provider: it redacts secrets before the provider sees them and blocks a team once it spends its cap. The **Tool Gateway** serves the agent's web search from a **tool kit** and can block a tool call by policy. Every decision is recorded in a trace.

## Setup
This cell loads your `.env`, connects to LangSmith, and shows which key each gateway will use. Gateway policies are created in the organization that owns `LANGSMITH_GATEWAY_API_KEY`, because that is where the gateway bills and governs your model calls. Tool kits live in this workspace and need a personal access token (`lsv2_pt_`) as `LANGSMITH_API_KEY`.

In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.config import environment_project  # sets the tracing destination before any LangSmith client exists
from langsmith import Client

from utils.gateway import describe_key, gateway_workspace_id
from utils.notebook import trace_link

client = Client()
dest = environment_project(client)
gateway_workspace = gateway_workspace_id()

print("LLM Gateway key: ", describe_key(os.environ["LANGSMITH_GATEWAY_API_KEY"]), "· workspace", gateway_workspace)
print("Tool Gateway key:", describe_key(os.environ["LANGSMITH_API_KEY"]), "· workspace", os.environ["LANGSMITH_WORKSPACE_ID"])
trace_link(dest.url, "Agent traces (Development)")

## 1. Route model calls through the LLM Gateway
Every model call in this tour already goes through the **LLM Gateway**: `utils/models.py` points the model at `https://gateway.smith.langchain.com/openai` and authenticates with `LANGSMITH_GATEWAY_API_KEY`. The gateway calls the provider with a key stored in that key's workspace, applies that organization's policies, and returns metadata about what it did on every response. The run below takes about 30 seconds.

> **Requires:** LLM Gateway (beta) for the gateway key's organization, a provider key or Gateway Credits there, and `gateway:invoke` on the key (Workspace Admin has it). **Without it:** read sections 1–3 and continue at section 4.

In [ ]:
from agents.client_research_agent import build_client_research_agent
from utils.gateway import GatewayInfoCollector, model_endpoint
from utils.models import make_model

tour_model = make_model()
agent = build_client_research_agent(model=tour_model)
collector = GatewayInfoCollector()
question = "What did NVIDIA announce this week? Search at most once."

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"configurable": {"thread_id": str(uuid4())}, "callbacks": [collector]},
)
print("Model endpoint:", model_endpoint(tour_model))
print("Answer:", result["messages"][-1].text[:300])
print("Model calls through the gateway:", len(collector.infos))
print("Outcomes:", sorted({info.get("outcome", "unknown") for info in collector.infos}))
trace_link(dest.url, "Agent traces (Development)")

Each model run in your agent's trace carries what the gateway reported about that call. The gateway also keeps its own trace of every call, in **LLM Gateway** → **Traces** of the organization that owns `LANGSMITH_GATEWAY_API_KEY`, which may not be this workspace. Each call is a row with tokens, cost and `gateway.credential_source`, but no request body, because trace content stays off until a data policy turns it on.

> For apps that build their own `ChatAnthropic` model or run Deep Agents, `LANGSMITH_GATEWAY=true` is the no-code switch: the model reads it when it is built and sends its calls to the gateway.

## 2. Redact secrets before they reach the provider
A **data policy** scans each outbound request, including the tool results the agent sends back, and replaces each match with a `[SAFE_TO_USE:<CATEGORY>_<id>]` placeholder. The gateway restores the real values in the response, so the provider never sees them and your agent never notices. This policy detects secrets plus one custom pattern, the firm's `ACCT-` account numbers. It applies only to requests that carry the header `X-Gateway-Tour: govern`, so nobody else's traffic changes.

> **Requires:** `organization:gateway:manage` in the gateway key's organization (Organization Admin or Operator). **Without it:** leave `create_policies = False`, and the policy cells print `Skipped`. Ask an admin to run sections 2–3, or read along.

In [ ]:
from utils.gateway import create_gateway_policy

create_policies = False  # an Organization Admin or Operator in the gateway key's org sets this to True
redaction_policy = {
    "name": "tour-redact-secrets",
    "policy_type": "guard",
    "action": "block",  # the only stored action; guard policies redact rather than block
    "subject_matchers": [
        {"key": "workspace_id", "value": gateway_workspace},
        {"key": "tour", "value": "govern"},  # matches the X-Gateway-Tour header
    ],
    "config": {
        "version": 1,
        "detect": {"pii": False, "secrets": True, "custom": [{"pattern": "ACCT-[0-9]{8}"}]},
        "trace": {"capture_content": True},
    },
}

redaction = create_gateway_policy(redaction_policy) if create_policies else None
if create_policies:
    print("Data policy:", redaction["name"] if redaction else "not created")
else:
    print("Data policy: Skipped (admin verifies live)")

In [ ]:
import re
import secrets
import string

fake_token = "ghp_" + "".join(secrets.choice(string.ascii_letters + string.digits) for _ in range(36))  # made up at runtime
leaky_prompt = (
    "Prep a one-paragraph meeting brief for client CL-0042 and include their account number. "
    f"From my call notes: they asked us to rotate the GitHub token {fake_token} we use for their reporting repo."
)

redacting_agent = build_client_research_agent(model=make_model(default_headers={"X-Gateway-Tour": "govern"}))
collector = GatewayInfoCollector()
result = redacting_agent.invoke(
    {"messages": [{"role": "user", "content": leaky_prompt}]},
    config={"configurable": {"thread_id": str(uuid4())}, "callbacks": [collector]},
)
answer = result["messages"][-1].text
redacted_calls = sum(1 for info in collector.infos if info.get("guard_redaction_applied"))
print("Redacted model calls:", redacted_calls, "of", len(collector.infos))
print("Account number in the answer:", bool(re.search(r"ACCT-[0-9]{8}", answer)))
trace_link(dest.url, "Agent traces (Development)")

Open **LLM Gateway** → **Traces** and click the newest row. The request shows `[SAFE_TO_USE:...]` placeholders where the token and the account number were, and `policy.matched_rules` lists the rules that fired. Your agent's own trace still shows the real values, because the gateway restores them before your code sees the response.

> System prompts and provider error bodies are neither scanned nor restored.

<img src="../images/TODO-06-gateway-trace-redacted.png" alt="LLM Gateway trace with SAFE_TO_USE placeholders in the request" style="width: auto; max-height: 360px; border-radius: 8px;">

## 3. Cap spend per team
A **spend policy** blocks a scope's requests with `402` once it has spent its cap. A header condition gives each team its own cap under one API key: here, a cap of a hundredth of a cent per hour on requests that send `X-Gateway-Team: advisory-desk`. The request that crosses the cap still succeeds, and the next one is blocked, so the cell after next runs the agent up to three times. Each run takes about 20 seconds.

> **Requires:** `organization:gateway:manage`, as in section 2. **Without it:** the cap cell prints `Skipped` and the runs below are all allowed. Read on for what a block looks like.

In [ ]:
spend_policy = {
    "name": "tour-advisory-desk-cap",
    "policy_type": "spend_cap",
    "action": "block",
    "subject_matchers": [
        {"key": "workspace_id", "value": gateway_workspace},
        {"key": "team", "value": "advisory-desk"},  # matches the X-Gateway-Team header
    ],
    "config": {"window": "hourly", "limit_usd": 0.0001},
}

spend_cap = create_gateway_policy(spend_policy) if create_policies else None
if create_policies:
    print("Spend cap:", f"{spend_cap['name']} ($0.0001/hour)" if spend_cap else "not created")
else:
    print("Spend cap: Skipped (admin verifies live)")

In [ ]:
import openai

desk_agent = build_client_research_agent(model=make_model(default_headers={"X-Gateway-Team": "advisory-desk"}))
desk_question = "In two sentences, what is a money market fund? Do not search."

for attempt in range(1, 4):
    try:
        desk_agent.invoke(
            {"messages": [{"role": "user", "content": desk_question}]},
            config={"configurable": {"thread_id": str(uuid4())}},
        )
        print(f"Run {attempt}: allowed")
    except openai.APIStatusError as exc:
        if exc.status_code != 402:
            raise
        print(f"Run {attempt}: blocked ({exc.status_code}) {exc.message}")
        break

In the agent's trace, the blocked run ends in a model run with a `402` error naming `tour-advisory-desk-cap`. In **LLM Gateway** → **Cost Controls**, the cap's **Usage** column is over 100%. Gateway policy violations also become issues in **Engine**, the same place Module 3's issue came from.

<img src="../images/TODO-06-cost-controls-usage.png" alt="Cost Controls table with the advisory-desk cap over 100% usage" style="width: auto; max-height: 320px; border-radius: 8px;">